# Tải và lọc nguồn

Tải từng version qua `/src/{id}vN`, nhận diện tar/gzip/TeX/PDF theo nội dung và chỉ giữ `.tex/.bib`. Lệnh figure trong TeX được giữ nguyên. Manifest phân biệt nguồn rỗng với lỗi tải.

Before là tổng byte chưa nén trong archive; after là byte `.tex/.bib` được giữ. Hai số trung bình dùng cùng tập paper có số đo hợp lệ.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from crawler.reporting import load_m1, run_table
m1 = load_m1()
papers, refs, versions, observations = (m1[k] for k in ('papers','references','versions','observations'))
print('Input:', m1['directory'])
print('Phạm vi:', len(papers), 'paper;', len(versions), 'version;', len(refs), 'working references')
print('Provenance:', m1['provenance'])


In [ ]:
status = versions['source_status'].fillna('unknown').value_counts()
display(status.to_frame('versions'))
status.plot.bar(title='Trạng thái nguồn theo version', ylabel='Số version'); plt.tight_layout(); plt.show()
valid = versions.dropna(subset=['size_before_bytes','size_after_bytes'])
valid = valid[valid['source_status'].eq('success')]
sizes = valid.groupby('arxiv_id')[['size_before_bytes','size_after_bytes']].sum()
if len(sizes):
    display(sizes.mean().to_frame('mean_bytes_per_paper_in_valid_set'))
    sizes.mean().plot.bar(title=f'Byte trung bình — cùng tập {len(sizes)} paper có TeX và số đo', ylabel='Byte')
    plt.tight_layout(); plt.show()
else:
    print('Chưa có tập hợp lệ để so sánh size; không thay null bằng 0.')


## Giới hạn

Source audit đối chiếu trạng thái success với tệp thật. PDF-only được báo riêng. Mức giảm do lọc được tính trên byte chưa nén ở cả hai phía.
